# AI-асистент продуктового аналітика
**Лабораторна робота №4.** AI-підкріплена дата аналітика · Методи та інструменти продуктової аналітики
**Виконав:** студент групи ІА-33 Супик Андрій

Асистент забирає на себе рутинну частину роботи аналітика: профілювання даних, пошук гіпотез, відповіді на питання природною мовою, розбір змін метрик і текст для стейкхолдера.

**Головний принцип:** усі числа рахує Pandas, а LLM (Google Gemini) лише інтерпретує вже пораховані факти й пише текст або код. Кожна відповідь моделі проходить програмну перевірку: чи існують згадані колонки й сегменти, чи збігаються числа з даними, чи безпечний згенерований код.

| № | Функція | Що робить |
|---|---|---|
| 1 | `auto_eda(df)` | Профіль таблиці (Pandas) + інтерпретація якості даних (LLM) |
| 2 | `suggest_hypotheses(users)` | 5–10 гіпотез, кожна посилається на факти з таблиці фактів |
| 3 | `ask("питання")` | Natural language → код Pandas → виконання → результат (код видно) |
| 4 | `explain_change(metric, ...)` | Розклад зміни метрики по сегментах + гіпотеза причини |
| 5 | `executive_summary(kpis)` | Короткий текст для стейкхолдера з перевіркою чисел |

**Датасет:** дані мобільного застосунку з ЛР3: `registrations` (202 тис. користувачів), `sessions` (1,5 млн сесій), `payments` (11 тис. оплат) і довідники.

## 0. Налаштування
**API-ключ.** Безкоштовний ключ Gemini: [aistudio.google.com](https://aistudio.google.com) → **Get API key → Create API key**.
У Colab найзручніше зберегти його в **Secrets** (іконка 🔑 зліва) з назвою `GEMINI_API_KEY` і увімкнути доступ для notebook. Якщо секрету немає, notebook попросить ввести ключ вручну (він не зберігається у файлі).

In [3]:
import os, re, json, time, ast, textwrap
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata          # Colab Secrets
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        GEMINI_API_KEY = None
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass('Встав Gemini API key: ')
print('Ключ отримано:', bool(GEMINI_API_KEY))

Ключ отримано: True


## 1. Клієнт LLM
Модель викликається через REST API Gemini, без додаткових бібліотек.
- **JSON mode** (`responseMimeType: application/json`): модель зобов'язана повернути JSON, тож відповідь розбирається як структура, а не витягується регулярками з тексту.
- **Низька температура (0.2)**: для аналітики потрібна стабільність, а не креативність.
- **Повтори з паузою** при перевантаженні (429/503) і **перехід на іншу модель**, якщо поточна недоступна.
- Усі виклики пишуться в `LLM_LOG`, щоб можна було подивитись промпт, відповідь і час.

In [4]:
PREFERRED_MODELS = ['gemini-4-hydrogen', 'gemini-flash-latest', 'gemini-2.5-flash', 'gemini-2.0-flash']


def pick_models():
    """Питає в API список доступних моделей і ставить першими ті, що є в PREFERRED_MODELS."""
    try:
        resp = requests.get('https://generativelanguage.googleapis.com/v1beta/models',
                            params={'key': GEMINI_API_KEY, 'pageSize': 200}, timeout=30)
        resp.raise_for_status()
        available = [m['name'].split('/')[-1] for m in resp.json().get('models', [])
                     if 'generateContent' in m.get('supportedGenerationMethods', [])]
    except Exception as e:
        print('Не вдалося отримати список моделей, використовую стандартний:', e)
        return PREFERRED_MODELS
    chosen = [m for m in PREFERRED_MODELS if m in available]
    # запасний варіант: будь-яка швидка модель Gemini з тих, що доступні
    chosen += [m for m in available if 'flash' in m and m not in chosen and 'image' not in m and 'tts' not in m][:3]
    return chosen or PREFERRED_MODELS


MODELS = pick_models()
print('Моделі (в порядку спроби):', MODELS)
API_URL = 'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent'
LLM_LOG = []


class LLMError(RuntimeError):
    pass


def parse_json(text):
    """Розбирає JSON із відповіді моделі (на випадок, якщо вона все ж обгорне його в ```json)."""
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise LLMError(f'Модель повернула не JSON: {text[:300]}')


def call_llm(system_prompt, user_prompt, temperature=0.2, max_retries=4):
    """Надсилає промпт у Gemini і повертає розібраний JSON."""
    body = {
        'systemInstruction': {'parts': [{'text': system_prompt}]},
        'contents': [{'role': 'user', 'parts': [{'text': user_prompt}]}],
        'generationConfig': {'temperature': temperature, 'responseMimeType': 'application/json'},
    }
    last_error = None
    for model in MODELS:
        for attempt in range(max_retries):
            started = time.time()
            try:
                resp = requests.post(API_URL.format(model=model), params={'key': GEMINI_API_KEY},
                                     json=body, timeout=180)
            except requests.RequestException as e:
                last_error = e
                time.sleep(3)
                continue
            if resp.status_code == 404:                     # модель недоступна → наступна
                last_error = f'{model}: 404'
                break
            if resp.status_code in (429, 500, 503):          # перевантаження → чекаємо й повторюємо
                last_error = f'{model}: {resp.status_code}'
                time.sleep(5 * (attempt + 1))
                continue
            if resp.status_code != 200:
                raise LLMError(f'{model}: HTTP {resp.status_code}: {resp.text[:300]}')
            data = resp.json()
            candidates = data.get('candidates') or []
            if not candidates or 'content' not in candidates[0]:
                raise LLMError(f'Порожня відповідь моделі: {json.dumps(data)[:300]}')
            text = ''.join(p.get('text', '') for p in candidates[0]['content'].get('parts', []))
            parsed = parse_json(text)
            LLM_LOG.append({'model': model, 'seconds': round(time.time() - started, 1),
                            'system': system_prompt, 'user': user_prompt, 'response': parsed})
            return parsed
    raise LLMError(f'Не вдалося отримати відповідь: {last_error}')


# Перевірка зв'язку
print(call_llm('Return JSON only.', 'Return {"status": "ok"}'))

Моделі (в порядку спроби): ['gemini-4-hydrogen', 'gemini-flash-latest', 'gemini-2.5-flash', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview']
{'status': 'ok'}


## 2. Завантаження даних
Файли лежать у папці `data/` репозиторію. Якщо notebook запущено в Colab і папки немає, репозиторій клонується автоматично. Великі таблиці стиснуті в `.csv.gz`, Pandas читає їх напряму.
Будуємо таблицю **`users`**: один рядок на користувача з атрибутами реєстрації та агрегатами по сесіях і оплатах. Саме вона є основою для гіпотез і питань.

In [5]:
DATA_DIR = 'data'
REPO_URL = 'https://github.com/insxlll/MIPA-4laba'   # репозиторій з папкою data/

if not os.path.exists(DATA_DIR):                      # у Colab даних ще немає → клонуємо репозиторій
    os.system(f'git clone -q --depth 1 {REPO_URL} repo_data')
    DATA_DIR = 'repo_data/data'


def read_table(name, **kwargs):
    for ext in ('.csv', '.csv.gz'):
        path = os.path.join(DATA_DIR, name + ext)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    raise FileNotFoundError(f'Немає файлу {name}.csv(.gz) у папці {DATA_DIR}/')


registrations = read_table('registrations', parse_dates=['reg_dt'])
payments = read_table('payments', parse_dates=['pay_dt'])
sessions = read_table('sessions', parse_dates=['session_start_dt'])
countries = read_table('country_info_dict').drop_duplicates('country_code')   # у довіднику є дубль коду
features = read_table('feature_type_dict')

payments = payments.merge(features, on='feature_type_id', how='left')

pay_by_user = payments.groupby('user_id').agg(
    revenue=('amount_usd', 'sum'), n_payments=('amount_usd', 'size'), first_pay_dt=('pay_dt', 'min'))
ses_by_user = sessions.groupby('user_id').agg(
    n_sessions=('session_start_dt', 'size'), total_duration_sec=('duration', 'sum'),
    first_session_dt=('session_start_dt', 'min'), last_session_dt=('session_start_dt', 'max'))
ses_by_user['total_duration_min'] = ses_by_user.pop('total_duration_sec') / 60

users = (registrations
         .merge(countries, on='country_code', how='left')
         .merge(pay_by_user, on='user_id', how='left')
         .merge(ses_by_user, on='user_id', how='left'))
users[['revenue', 'n_payments', 'n_sessions', 'total_duration_min']] = \
    users[['revenue', 'n_payments', 'n_sessions', 'total_duration_min']].fillna(0)
users['is_payer'] = users['n_payments'] > 0
users['age_group'] = pd.cut(users['age'], [17, 24, 34, 44, 54, 120],
                            labels=['18-24', '25-34', '35-44', '45-54', '55+'])
users['is_new'] = users['reg_dt'] >= '2024-04-05'      # зареєстровані у періоді даних

TABLES = {'users': users, 'registrations': registrations, 'sessions': sessions,
          'payments': payments, 'countries': countries}
for name, df in TABLES.items():
    print(f'{name:14s} {len(df):>10,} рядків × {df.shape[1]} колонок')

users             202,382 рядків × 17 колонок
registrations     202,382 рядків × 6 колонок
sessions        1,507,685 рядків × 4 колонок
payments           11,330 рядків × 5 колонок
countries             252 рядків × 2 колонок


## 3. Перевірки проти галюцинацій
LLM упевнено вигадує числа, яких немає в даних. Тому:
- `check_numbers(text, facts)` витягує всі числа з тексту моделі й шукає їх серед фактично порахованих значень (з урахуванням округлення й відсотків). Числа, яких немає в даних, позначаються ⚠️.
- у функціях нижче окремо перевіряється, що згадані колонки, сегменти та ідентифікатори фактів **реально існують**.

In [6]:
def _collect_numbers(obj, out):
    if isinstance(obj, dict):
        for v in obj.values():
            _collect_numbers(v, out)
    elif isinstance(obj, (list, tuple, set)):
        for v in obj:
            _collect_numbers(v, out)
    elif isinstance(obj, (int, float, np.integer, np.floating)) and not isinstance(obj, bool):
        if np.isfinite(obj):
            out.append(float(obj))
    elif isinstance(obj, str):
        for n in _extract_numbers(obj):
            out.append(n)
    return out


def _extract_numbers(text):
    text = str(text)
    # дати й час не є «числами» для перевірки
    text = re.sub(r'\d{4}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}(?::\d{2})?)?', ' ', text)
    text = re.sub(r'\b\d{1,2}\.\d{1,2}\.\d{4}\b', ' ', text)
    pattern = r'(?<![\w.])-?\d{1,3}(?:[ \u00a0]\d{3})+(?![\d])(?:[.,]\d+)?|(?<![\w.])-?\d+(?:[.,]\d+)*'
    nums = []
    for raw in re.findall(pattern, text):
        s = raw.replace('\u00a0', '').replace(' ', '')
        if ',' in s and '.' in s:
            s = s.replace(',', '')
        elif s.count(',') == 1 and len(s.split(',')[1]) != 3:
            s = s.replace(',', '.')
        else:
            s = s.replace(',', '')
        if s.count('.') > 1:
            continue
        try:
            nums.append(float(s))
        except ValueError:
            pass
    return nums


def check_numbers(text, facts, rel_tol=0.02, ignore=(2024, 2025, 2026)):
    """Повертає числа з тексту, яких немає серед фактів (можливі галюцинації)."""
    known = _collect_numbers(facts, [])
    variants = set()
    for k in known:
        variants.update({k, k * 100, k / 1000, k / 1_000_000, round(k), round(k * 100)})
    unverified = []
    for n in _extract_numbers(text):
        if abs(n) <= 10 and float(n).is_integer():      # дрібні цілі: номери пунктів, «топ-5», дні
            continue
        if n in ignore:
            continue
        if not any(abs(n - v) <= max(rel_tol * abs(v), 0.051) for v in variants):
            unverified.append(n)
    return unverified


def show_check(text, facts, label='Перевірка чисел'):
    bad = check_numbers(text, facts)
    if bad:
        display(Markdown(f'⚠️ **{label}:** числа, яких немає в даних: `{bad}`. Їх треба перевірити вручну.'))
    else:
        display(Markdown(f'✅ **{label}:** усі числа в тексті знайдено серед пораховано даних.'))
    return bad

## Функція 1. Автоматичний EDA
**Pandas рахує** профіль кожної колонки: тип, пропуски, кількість унікальних, статистики, викиди за правилом 3×IQR, від'ємні й нульові значення, діапазон дат, пробіли та різний регістр у тексті.
**LLM отримує лише цей профіль** (не сирі дані) і пише висновки про якість даних у форматі JSON.
**Перевірка:** проблеми, що посилаються на неіснуючі колонки, відкидаються; числа в тексті звіряються з профілем.

In [7]:
EDA_SYSTEM = """You are a senior product data analyst reviewing data quality.
You receive a JSON profile of ONE table computed with pandas. You never see raw rows.
Rules:
- Use ONLY numbers that appear in the profile. Never invent or recompute numbers.
- Every issue must reference an existing column name from the profile.
- Be specific: say which column, what is wrong, and the evidence from the profile.
- Severity: "high" breaks metrics, "medium" may bias results, "low" is cosmetic.
- Write all text in Ukrainian.
Return JSON exactly in this schema:
{"summary": "2-3 sentences about the table and its overall quality",
 "issues": [{"column": "...", "problem": "...", "evidence": "...", "severity": "high|medium|low", "fix": "..."}],
 "observations": ["notable facts about distributions, 3-5 items"],
 "next_steps": ["what an analyst should check next, 2-4 items"]}"""


def profile_table(df, max_top=5):
    prof = {'rows': int(len(df)), 'columns': int(df.shape[1]),
            'duplicate_rows': int(df.duplicated().sum()) if len(df) else 0, 'columns_profile': {}}
    for col in df.columns:
        s = df[col]
        p = {'dtype': str(s.dtype), 'missing': int(s.isna().sum()),
             'missing_pct': round(float(s.isna().mean() * 100), 2) if len(s) else 0.0,
             'unique': int(s.nunique(dropna=True))}
        if pd.api.types.is_bool_dtype(s):
            p['true_pct'] = round(float(s.mean() * 100), 2) if len(s) else None
        elif pd.api.types.is_numeric_dtype(s):
            d = s.dropna()
            if len(d):
                q1, q3 = d.quantile([0.25, 0.75])
                iqr = q3 - q1
                p.update({'min': float(d.min()), 'p25': float(q1), 'median': float(d.median()),
                          'p75': float(q3), 'max': float(d.max()), 'mean': round(float(d.mean()), 3),
                          'negative': int((d < 0).sum()), 'zeros': int((d == 0).sum()),
                          'outliers_3iqr': int(((d < q1 - 3 * iqr) | (d > q3 + 3 * iqr)).sum()) if iqr > 0 else 0})
        elif pd.api.types.is_datetime64_any_dtype(s):
            d = s.dropna()
            if len(d):
                p.update({'min': str(d.min()), 'max': str(d.max()),
                          'top_years': {str(k): int(v) for k, v in d.dt.year.value_counts().head(3).items()}})
        else:
            d = s.dropna().astype(str)
            p['top_values'] = {k: int(v) for k, v in d.value_counts().head(max_top).items()}
            p['values_with_spaces'] = int((d != d.str.strip()).sum())
            p['case_variants'] = int(d.nunique() - d.str.lower().str.strip().nunique())
        prof['columns_profile'][col] = p
    return prof


def auto_eda(df, name='table'):
    """Профіль таблиці (Pandas) + інтерпретація якості даних (LLM)."""
    if df is None or len(df) == 0:
        display(Markdown(f'### EDA: `{name}`\n❌ Таблиця порожня, аналізувати нічого. LLM не викликається.'))
        return None
    prof = profile_table(df)
    display(Markdown(f'### EDA: `{name}` — {prof["rows"]:,} рядків × {prof["columns"]} колонок, '
                     f'дублікатів: {prof["duplicate_rows"]:,}'))
    display(pd.DataFrame(prof['columns_profile']).T.fillna(''))

    report = call_llm(EDA_SYSTEM, f'Table name: {name}\nProfile JSON:\n{json.dumps(prof, ensure_ascii=False, default=str)}')

    issues = [i for i in report.get('issues', []) if i.get('column') in df.columns]
    rejected = [i for i in report.get('issues', []) if i.get('column') not in df.columns]

    md = [f'**Висновок:** {report.get("summary", "")}', '', '**Проблеми якості даних:**']
    icon = {'high': '🔴', 'medium': '🟠', 'low': '🟡'}
    for i in issues:
        md.append(f'- {icon.get(i.get("severity"), "•")} `{i["column"]}`: {i.get("problem", "")} '
                  f'*Доказ:* {i.get("evidence", "")} *Що зробити:* {i.get("fix", "")}')
    md += ['', '**Спостереження:**'] + [f'- {o}' for o in report.get('observations', [])]
    md += ['', '**Що перевірити далі:**'] + [f'- {n}' for n in report.get('next_steps', [])]
    display(Markdown('\n'.join(md)))
    if rejected:
        display(Markdown(f'⚠️ Відкинуто {len(rejected)} проблем(и) з неіснуючими колонками: '
                         f'{[i.get("column") for i in rejected]}'))
    show_check(json.dumps(report, ensure_ascii=False), prof)
    return {'profile': prof, 'report': report}

In [8]:
eda_users = auto_eda(registrations, 'registrations')

### EDA: `registrations` — 202,382 рядків × 6 колонок, дублікатів: 0

/tmp/ipykernel_4107/1414619107.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(pd.DataFrame(prof['columns_profile']).T.fillna(''))


,dtype,missing,missing_pct,unique,min,p25,median,p75,max,mean,negative,zeros,outliers_3iqr,top_years,top_values,values_with_spaces,case_variants
user_id,int64,0,0.00,202382,"20,001.00","260,016,203.00","267,898,286.50","268,630,980.25","269,309,916.00","244,008,510.08",0,0,24198,,,,
reg_dt,datetime64[ns],0,0.00,198524,2013-04-05 05:16:00,,,,2024-05-11 11:04:47,,,,,"{'2024': 171866, '2023': 22782, '2022': 5481}",,,
gender,object,0,0.00,2,,,,,,,,,,,"{'m': 200109, 'f': 2273}",0,0
age,float64,2,0.00,72,18.00,27.00,36.00,45.00,89.00,36.96,0,0,0,,,,
app,object,0,0.00,5,,,,,,,,,,,"{'mobile': 188744, 'ios': 8836, 'desktop': 353...",0,0
country_code,object,1,0.00,195,,,,,,,,,,,"{'US': 31842, 'UA': 25467, 'ZA': 20923, 'PE': ...",0,0


**Висновок:** Таблиця registrations містить 202382 рядки та 6 колонок без повних дублікатів (duplicate_rows: 0) і з мінімальною кількістю пропусків. Загальна якість даних прийнятна, проте метрики можуть суттєво спотворювати змішані категорії платформ у app та сильний перекіс у gender. Також варто враховувати тривалий історичний діапазон у reg_dt (від 2013-04-05 05:16:00 до 2024-05-11 11:04:47) та викиди у user_id.

**Проблеми якості даних:**
- 🔴 `app`: Змішана таксономія платформ: загальна категорія mobile співіснує з конкретними платформами ios та android, а також є значення unknown. *Доказ:* У колонці 5 унікальних значень, серед яких top_values: mobile (188744), ios (8836), desktop (3539), android (1260), unknown (3). *Що зробити:* Уточнити визначення категорії mobile (наприклад, мобільний веб чи застаріле групування), розділити або уніфікувати мапінг із ios та android і перевірити записи зі значенням unknown.
- 🟠 `gender`: Екстремальний дисбаланс значень статі на користь категорії m, що може свідчити про значення за замовчуванням у формі реєстрації або специфічний/ботовий трафік. *Доказ:* При unique: 2 та missing: 0 значення m зустрічається 200109 разів, тоді як f — лише 2273 рази із 202382 рядків. *Що зробити:* Перевірити логіку вибору статі під час реєстрації (чи не підставляється m автоматично) та сегментувати аномалію за датами й країнами.
- 🟠 `reg_dt`: Дуже широкий історичний діапазон дат із різкою концентрацією в останніх роках та наявністю однакових міток часу. *Доказ:* Діапазон від min: 2013-04-05 05:16:00 до max: 2024-05-11 11:04:47, але у top_years переважають 2024 (171866), 2023 (22782) та 2022 (5481); unique становить 198524 із 202382 рядків. *Що зробити:* Перевірити коректність дат старих акаунтів (починаючи з 2013-04-05 05:16:00) та переконатися, що повторювані мітки часу не є наслідком пакетної міграції.
- 🟡 `user_id`: Велика кількість статистичних викидів у бік менших значень, що може вказувати на зміну схеми генерації ID або об'єднання різних баз. *Доказ:* Кількість outliers_3iqr дорівнює 24198; min становить 20001.0 при p25: 260016203.0, median: 267898286.5, max: 269309916.0 та mean: 244008510.08. *Що зробити:* Перевірити зв'язок менших значень user_id зі старими датами у reg_dt та задокументувати зміну діапазонів ідентифікаторів.
- 🟡 `age`: Наявність пропусків і зберігання цілих значень віку у дробовому типі даних float64. *Доказ:* Колонка має dtype: float64 та missing: 2 (missing_pct: 0.0) при діапазоні від min: 18.0 до max: 89.0. *Що зробити:* Конвертувати тип даних у цілочисельний із підтримкою пропусків (наприклад, Int64) та перевірити 2 записи без зазначеного віку.
- 🟡 `country_code`: Поодинокий пропуск коду країни. *Доказ:* У колонці зафіксовано missing: 1 (missing_pct: 0.0) із 202382 рядків. *Що зробити:* Знайти рядок із пропущеним country_code та за можливості відновити код країни з технічних логів або позначити як невизначений.

**Спостереження:**
- Таблиця містить 202382 рядки та 6 колонок без повних дублікатів (duplicate_rows: 0), а колонка user_id є повністю унікальною (unique: 202382, missing: 0, negative: 0, zeros: 0).
- Найбільше реєстрацій у top_years зафіксовано у 2024 році (171866), тоді як у 2023 році їх було 22782, а у 2022 році — 5481.
- Розподіл age є стабільним без викидів (outliers_3iqr: 0): значення лежать у межах від 18.0 до 89.0 з median: 36.0, p25: 27.0, p75: 45.0 та mean: 36.961.
- Колонка country_code охоплює 195 унікальних значень без пробілів і варіантів регістру (values_with_spaces: 0, case_variants: 0); серед лідерів US (31842), UA (25467), ZA (20923), PE (11943) та DO (11906).

**Що перевірити далі:**
- Перевірити зв'язок між 24198 викидами у user_id та роками реєстрації у reg_dt, щоб з'ясувати, чи це легасі-користувачі з попередніх систем.
- Узгодити з продуктовою або інженерною командою семантику значення mobile у колонці app у порівнянні з ios та android.
- Проаналізувати розподіл gender (200109 значень m проти 2273 значень f) у розрізі top_years та country_code, щоб відокремити можливу помилку збору даних від реального профілю аудиторії.

⚠️ **Перевірка чисел:** числа, яких немає в даних: `[2023.0, 2022.0, 2023.0, 2022.0]`. Їх треба перевірити вручну.

In [9]:
eda_sessions = auto_eda(sessions, 'sessions')

### EDA: `sessions` — 1,507,685 рядків × 4 колонок, дублікатів: 14

/tmp/ipykernel_4107/1414619107.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(pd.DataFrame(prof['columns_profile']).T.fillna(''))


,dtype,missing,missing_pct,unique,min,p25,median,p75,max,mean,negative,zeros,outliers_3iqr,top_years
user_id,int64,0,0.00,196319,"20,001.00","245,273,060.00","262,563,511.00","267,795,910.00","269,309,863.00","240,574,612.26",0,0,128222,
session_start_dt,datetime64[ns],0,0.00,1003307,2024-04-05 00:21:24,,,,2024-05-11 11:05:06,,,,,{'2024': 1507685}
duration,int64,0,0.00,13027,0.00,"1,244.00","1,373.00","1,832.00","71,108.00","1,805.27",0,26,83104,
session_number,int64,0,0.00,13612,1.00,12.00,120.00,691.00,"35,884.00",694.92,0,0,98719,


**Висновок:** Таблиця sessions містить 1507685 рядків та 4 колонки без жодних пропусків (missing: 0, missing_pct: 0.0) за період з 2024-04-05 00:21:24 до 2024-05-11 11:05:06. Загальна технічна якість даних висока, проте виявлено 14 повних дублікатів рядків (duplicate_rows: 14) та значну кількість викидів у числових колонках. Особливу увагу слід звернути на нульові та екстремально великі значення тривалості сесій і аномально високі номери сесій, які можуть спотворювати продуктові метрики.

**Проблеми якості даних:**
- 🔴 `session_number`: Аномально високі порядкові номери сесій та сильний правий перекіс розподілу, що може свідчити про наявність ботів, автоматизованого трафіку або помилки інкрементації лічильника сесій. *Доказ:* max: 35884.0 при median: 120.0 та p75: 691.0, mean: 694.923, а кількість викидів становить outliers_3iqr: 98719. *Що зробити:* Перевірити користувачів із високими значеннями session_number на наявність ботів або збоїв трекінгу та за потреби відфільтрувати їх перед розрахунком метрик.
- 🟠 `duration`: Наявність екстремально довгих сесій, що суттєво зміщує середню тривалість сесії вгору. *Доказ:* max: 71108.0 при p75: 1832.0 та median: 1373.0, mean: 1805.271, зафіксовано outliers_3iqr: 83104. *Що зробити:* Дослідити природу сесій із великою тривалістю (можливі незакриті сесії або фонові процеси) та використовувати median або застосовувати обмеження верхньої межі при аналізі.
- 🟡 `duration`: Присутні сесії з нульовою тривалістю, які можуть бути некоректно залогованими або миттєво перерваними подіями. *Доказ:* min: 0.0, кількість нульових значень zeros: 26. *Що зробити:* Перевірити логіку трекінгу цих 26 записів та вирішити, чи варто виключати їх під час обчислення середньої тривалості сесій.
- 🟡 `user_id`: Значний лівий перекіс ідентифікаторів користувачів, що вказує на наявність окремої групи дуже малих ID (імовірно, давні або тестові акаунти). *Доказ:* min: 20001.0 при p25: 245273060.0 та median: 262563511.0, через що mean: 240574612.257 нижче за p25, а кількість викидів складає outliers_3iqr: 128222. *Що зробити:* Звірити діапазон малих user_id зі словником користувачів, щоб переконатися, що це не тестові чи службові облікові записи.

**Спостереження:**
- У таблиці налічується 196319 унікальних значень user_id серед 1507685 рядків без від'ємних чи нульових значень (negative: 0, zeros: 0).
- Усі записи в session_start_dt належать до 2024 року (top_years: 2024 — 1507685), водночас кількість унікальних міток часу становить 1003307.
- Розподіл duration є правосторонньо асиметричним: значення змінюються від min: 0.0 до max: 71108.0, при цьому p25: 1244.0, median: 1373.0, p75: 1832.0, а mean: 1805.271.
- Колонка session_number починається з min: 1.0 і має 13612 унікальних значень, де median: 120.0 значно менша за mean: 694.923 та p75: 691.0.

**Що перевірити далі:**
- Знайти та видалити 14 повних дублікатів рядків (duplicate_rows: 14), а також перевірити унікальність комбінації user_id, session_start_dt та session_number.
- Проаналізувати сесії з аномально великими значеннями session_number (до max: 35884.0) та duration (до max: 71108.0), щоб з'ясувати, чи належать вони одним і тим самим користувачам.
- Перевірити користувачів із низькими значеннями user_id (поблизу min: 20001.0, що потрапили у outliers_3iqr: 128222) на предмет тестового трафіку або змін у системі генерації ID.
- Уточнити з командою розробки причину появи 26 записів із duration рівним 0.0.

✅ **Перевірка чисел:** усі числа в тексті знайдено серед пораховано даних.

## Функція 2. Генерація гіпотез
**Pandas будує таблицю фактів:** конверсію в оплату, ARPU, сесії й тривалість у розрізах платформи, статі, віку, країни, тижня реєстрації та дня тижня. Кожен факт має свій `fact_id`.
**LLM пропонує гіпотези й посилається тільки на `fact_id`**, а не пише числа сам. Числа в доказах підставляються з таблиці фактів програмно, тому вигадати їх модель не може. Гіпотези з неіснуючими `fact_id` відкидаються.

In [10]:
HYPOTHESES_SYSTEM = """You are a senior product analyst. You receive a table of facts computed with pandas
about a freemium mobile app (users, sessions, payments). Each fact has a fact_id.
Task: propose {n} hypotheses worth investigating.
Rules:
- Each hypothesis must be supported by 1-3 fact_ids from the table. Cite ONLY existing fact_ids.
- Do not write numbers yourself: the system inserts the values for the cited fact_ids.
- Compare a segment with its baseline (fact rows with segment "ALL") or with other segments.
- Prefer big, actionable differences; ignore segments with fewer than 500 users.
- Mention possible data-quality explanations when a pattern looks like a tracking artifact.
- Write all text in Ukrainian.
Return JSON: {"hypotheses": [{"title": "short", "hypothesis": "1-2 sentences",
 "fact_ids": ["F001"], "why_it_matters": "...", "how_to_check": "concrete analysis to run",
 "priority": "high|medium|low"}]}"""


def build_fact_table(u, s, min_users=500):
    rows = []

    def add(dimension, segment, metric, value, n_users):
        rows.append({'dimension': dimension, 'segment': str(segment), 'metric': metric,
                     'value': round(float(value), 4), 'n_users': int(n_users)})

    def seg_metrics(df, dimension, segment):
        n = len(df)
        add(dimension, segment, 'users', n, n)
        add(dimension, segment, 'payer_rate_pct', df['is_payer'].mean() * 100, n)
        add(dimension, segment, 'arpu_usd', df['revenue'].mean(), n)
        add(dimension, segment, 'sessions_per_user', df['n_sessions'].mean(), n)
        add(dimension, segment, 'minutes_per_user', df['total_duration_min'].mean(), n)

    seg_metrics(u, 'overall', 'ALL')
    for dim in ['app', 'gender', 'age_group', 'country_name', 'is_new']:
        for seg, part in u.groupby(dim, observed=True):
            if len(part) >= min_users:
                seg_metrics(part, dim, seg)

    new = u[u['is_new']]
    for week, part in new.groupby(new['reg_dt'].dt.to_period('W').dt.start_time):
        add('reg_week', week.date(), 'users', len(part), len(part))
        add('reg_week', week.date(), 'payer_rate_pct', part['is_payer'].mean() * 100, len(part))

    daily = s.groupby(s['session_start_dt'].dt.date)['user_id'].nunique()
    by_dow = daily.groupby(pd.to_datetime(daily.index).day_name()).mean()
    for dow, v in by_dow.items():
        add('weekday', dow, 'avg_dau', v, v)

    facts = pd.DataFrame(rows)
    facts.insert(0, 'fact_id', [f'F{i:03d}' for i in range(1, len(facts) + 1)])
    return facts


def suggest_hypotheses(u=None, s=None, n=7):
    u = users if u is None else u
    s = sessions if s is None else s
    if len(u) == 0:
        display(Markdown('❌ Немає даних для гіпотез.'))
        return None
    facts = build_fact_table(u, s)
    display(Markdown(f'**Таблиця фактів:** {len(facts)} фактів (показано перші 15)'))
    display(facts.head(15))

    result = call_llm(HYPOTHESES_SYSTEM.replace('{n}', str(n)), 'Facts:\n' + facts.to_csv(index=False))
    fact_index = facts.set_index('fact_id')

    md, kept, dropped = [], [], []
    for h in result.get('hypotheses', []):
        ids = [f for f in h.get('fact_ids', []) if f in fact_index.index]
        missing = [f for f in h.get('fact_ids', []) if f not in fact_index.index]
        if not ids:
            dropped.append(h.get('title'))
            continue
        kept.append(h)
        evidence = '; '.join(
            f'{fact_index.loc[f, "dimension"]}={fact_index.loc[f, "segment"]}: '
            f'{fact_index.loc[f, "metric"]} = {fact_index.loc[f, "value"]:,.2f}' for f in ids)
        md.append(f'### {len(kept)}. {h.get("title", "")}  `{h.get("priority", "")}`\n'
                  f'{h.get("hypothesis", "")}\n\n'
                  f'- **Факти з даних (підставлено програмно):** {evidence}\n'
                  f'- **Чому важливо:** {h.get("why_it_matters", "")}\n'
                  f'- **Як перевірити:** {h.get("how_to_check", "")}'
                  + (f'\n- ⚠️ Відкинуто неіснуючі fact_id: {missing}' if missing else ''))
    display(Markdown('\n\n'.join(md)))
    if dropped:
        display(Markdown(f'⚠️ Відкинуто гіпотези без жодного реального факту: {dropped}'))
    show_check(json.dumps(kept, ensure_ascii=False), facts['value'].tolist(), 'Перевірка чисел у тексті гіпотез')
    return {'facts': facts, 'hypotheses': kept}

In [11]:
hyp = suggest_hypotheses(n=7)

**Таблиця фактів:** 274 фактів (показано перші 15)

,fact_id,dimension,segment,metric,value,n_users
0,F001,overall,ALL,users,"202,382.00",202382
1,F002,overall,ALL,payer_rate_pct,3.84,202382
2,F003,overall,ALL,arpu_usd,0.46,202382
3,F004,overall,ALL,sessions_per_user,7.45,202382
4,F005,overall,ALL,minutes_per_user,224.15,202382
5,F006,app,android,users,"1,260.00",1260
6,F007,app,android,payer_rate_pct,12.14,1260
7,F008,app,android,arpu_usd,1.91,1260
8,F009,app,android,sessions_per_user,15.61,1260
9,F010,app,android,minutes_per_user,501.64,1260


### 1. Аномалія трекінгу платформ та розрив монетизації між Android і Mobile  `high`
Сегмент Android демонструє аномально високу частку платників порівняно із загальним базовим рівнем та сегментом Mobile, який охоплює переважну більшість аудиторії. Співіснування категорії Mobile поряд із конкретними ОС вказує на артефакт атрибуції або помилку класифікації, наприклад, коли події Android фіксуються лише після авторизації чи оплати або Mobile позначає мобільний веб.

- **Факти з даних (підставлено програмно):** app=android: payer_rate_pct = 12.14; app=mobile: payer_rate_pct = 3.70; overall=ALL: payer_rate_pct = 3.84
- **Чому важливо:** Якщо розрив спричинений технічною помилкою в зборі даних, команда може хибно оцінювати ефективність нативного застосунку; якщо ж проблема реальна, мобільний потік втрачає величезний обсяг потенційного доходу.
- **Як перевірити:** Побудувати матрицю перетинів поля платформи з User-Agent, версією SDK та подіями життєвого циклу, перевірити дублювання користувачів між платформами та порівняти воронку від першого візиту до оплати в розрізі типів пристроїв.

### 2. Парадокс високої активності та низької конверсії серед давніх користувачів  `high`
Існуючі користувачі генерують значно більше сесій на особу, ніж нові, проте їхній показник конверсії у платіж суттєво нижчий за конверсію новачків. Це може пояснюватися агресивним пейволом під час онбордингу нових користувачів або ж артефактом обрізаного вікна спостереження, де попередні покупки давніх користувачів не потрапили до вибірки.

- **Факти з даних (підставлено програмно):** is_new=False: payer_rate_pct = 1.87; is_new=True: payer_rate_pct = 4.73; is_new=False: sessions_per_user = 14.49
- **Чому важливо:** Розуміння того, чи приносять давні користувачі повторний дохід, чи просто споживають серверні ресурси безкоштовно, визначає стратегію підписок і утримання.
- **Як перевірити:** Перевірити повну історію транзакцій за весь життєвий цикл для сегмента існуючих користувачів, відокремити активні підписки від разових покупок та оцінити частку скинутих ідентифікаторів при перевстановленні.

### 3. Гендерний перекіс вибірки та вищий ARPU серед жіночої аудиторії  `medium`
Кількість користувачів жіночої статі є мізерною порівняно з чоловічою базою, але жіночий сегмент приносить суттєво вищий середній дохід на користувача. Такий різкий демографічний дисбаланс часто є наслідком дефекту форми реєстрації, коли стать за замовчуванням записується як чоловіча, а жіночу мають лише ті, хто свідомо заповнив профіль.

- **Факти з даних (підставлено програмно):** gender=f: users = 2,273.00; gender=f: arpu_usd = 0.76; gender=m: arpu_usd = 0.46
- **Чому важливо:** Некоректне значення за замовчуванням спотворює маркетингове таргетування, а реальна висока платоспроможність жіночої аудиторії може бути недооціненою точкою зростання.
- **Як перевірити:** Проаналізувати розподіл статі залежно від способу реєстрації та джерела трафіку, перевірити наявність значень за замовчуванням у бекенд-схемі профілю та порівняти ARPU між тими, хто заповнював анкету, і тими, хто її пропустив.

### 4. Розрив між залученістю та оплатою в крайніх вікових групах  `medium`
Найстарша вікова група проводить у застосунку в рази більше сесій, ніж наймолодша, проте демонструє найнижчу частку платників серед усіх вікових категорій. Висока кількість сесій без оплати може свідчити як про складність платіжного інтерфейсу для старших користувачів, так і про фонове або спільне використання акаунтів.

- **Факти з даних (підставлено програмно):** age_group=18-24: sessions_per_user = 2.72; age_group=55+: sessions_per_user = 15.13; age_group=55+: payer_rate_pct = 3.10
- **Чому важливо:** Утримання та висока залученість старшого сегмента вже досягнуті, тому усунення бар'єрів на етапі оплати здатне швидко збільшити дохід, тоді як серед молоді критичніше покращити базовий онбординг.
- **Як перевірити:** Дослідити тривалість та час доби сесій, проаналізувати кроки відмови на платіжному екрані за віковими групами, а також перевірити аномальну кількість пристроїв на один обліковий запис у старшому сегменті.

### 5. Надвисока залученість у Польщі без пропорційної монетизації  `high`
Користувачі в Польщі демонструють рекордно високу кількість сесій на користувача, але їхня конверсія в оплату суттєво відстає від сусідньої України та інших сильних європейських ринків. Це може бути спричинено відсутністю звичних локальних платіжних методів і валюти або ж викривленням метрик через ботовий трафік і парсинг.

- **Факти з даних (підставлено програмно):** country_name=Poland: sessions_per_user = 22.06; country_name=Poland: payer_rate_pct = 3.33; country_name=Ukraine: payer_rate_pct = 8.58
- **Чому важливо:** Якщо активність у Польщі органічна, адаптація локальних методів оплати та цін розблокує великий потенціал доходу; якщо це фрод, потрібно очистити аналітику та заблокувати аномальний трафік.
- **Як перевірити:** Порівняти розподіл сесій на користувача в Польщі для виявлення викидів, перевірити структуру IP-адрес і версій клієнта, а також оцінити успішність платіжних спроб за локальними провайдерами.

### 6. Низька конверсія на великих ринках Латинської Америки  `medium`
Великі за обсягом аудиторії країни Латинської Америки, зокрема Домініканська Республіка та Перу, показують у кілька разів нижчу частку платників порівняно із загальним базовим рівнем. Основною причиною ймовірно є відсутність адаптованого ціноутворення за паритетом купівельної спроможності або незручність міжнародних банківських карток для цих регіонів.

- **Факти з даних (підставлено програмно):** country_name=Dominican Republic: payer_rate_pct = 1.35; country_name=Peru: payer_rate_pct = 1.82; overall=ALL: payer_rate_pct = 3.84
- **Чому важливо:** Ці країни формують значну частку користувацької бази, тому навіть помірне зростання конверсії через локалізацію тарифів забезпечить відчутний приріст виручки.
- **Як перевірити:** Проаналізувати частку відхилених транзакцій та причини помилок оплати в цих країнах, порівняти поточні ціни із середнім чеком конкурентів на локальному ринку та спроєктувати A/B-тест знижених тарифних планів.

### 7. Зниження конверсії когорт на тлі різкого масштабування реєстрацій  `high`
Різкий сплеск тижневих реєстрацій наприкінці квітня супроводжувався помітним зниженням частки платників у травневій когорті порівняно з піковою конверсією попередньої когорти. Така динаміка може свідчити як про погіршення якості залученого трафіку під час масштабування, так і про ефект недозрівання даних, коли новіші користувачі ще не встигли здійснити покупку.

- **Факти з даних (підставлено програмно):** reg_week=2024-04-22: payer_rate_pct = 5.27; reg_week=2024-04-29: users = 41,340.00; reg_week=2024-05-06: payer_rate_pct = 4.15
- **Чому важливо:** Оцінка свіжих когорт без вирівнювання часового вікна призводить до хибних рішень щодо вимкнення або масштабування рекламних кампаній.
- **Як перевірити:** Побудувати когортну таблицю конверсії за однаковими часовими вікнами після дати реєстрації та розбити динаміку за каналами залучення й країнами, щоб відокремити часовий лаг від падіння якості трафіку.

✅ **Перевірка чисел у тексті гіпотез:** усі числа в тексті знайдено серед пораховано даних.

## Функція 3. Natural Language → Pandas
Користувач питає звичайною мовою. LLM отримує **схему таблиць** (колонки, типи, 3 приклади рядків) і повертає код Pandas.
- **Код завжди показується**, щоб аналітик бачив логіку.
- **Перевірка безпеки (AST):** заборонені `import`, доступ до файлів, мережі, `eval/exec`, «приховані» атрибути `__...__`. Небезпечний код не виконується.
- Код виконується на **копіях** таблиць, тому оригінальні дані не зміняться.
- Якщо код падає з помилкою, модель отримує текст помилки й **один раз виправляє** код сама.
- Якщо питання не про дані, модель повертає `can_answer: false` і пояснення.

In [12]:
NL2PANDAS_SYSTEM = """You translate analytics questions into pandas code.
Available pandas DataFrames (already loaded) and their schema are given below.
Rules:
- Use only pandas (pd), numpy (np) and the listed DataFrames. No imports, no file or network access, no plotting.
- Do not modify the DataFrames in place; build new objects.
- Store the final answer in a variable named result (a DataFrame, Series or scalar). Keep it small: at most 50 rows.
- Use the exact column names from the schema. Dates are pandas datetime64.
- If the question cannot be answered with these tables (or is not about the data), set can_answer to false and explain why.
- Write explanation and assumptions in Ukrainian.
Return JSON: {"can_answer": true, "code": "...", "explanation": "what the code does, 1-2 sentences", "assumptions": ["..."]}

Schema:
{schema}"""

INTERPRET_SYSTEM = """You are a product analyst. Given a question and the computed result (from pandas),
write a short answer in Ukrainian (2-3 sentences). Use only numbers present in the result. Return JSON: {"answer": "..."}"""

FORBIDDEN_NAMES = {'open', 'exec', 'eval', 'compile', '__import__', 'globals', 'locals', 'vars',
                   'getattr', 'setattr', 'delattr', 'input', 'os', 'sys', 'subprocess', 'shutil',
                   'requests', 'pathlib', 'builtins', 'breakpoint'}
FORBIDDEN_ATTRS = {'to_csv', 'to_excel', 'to_pickle', 'to_parquet', 'to_sql', 'read_csv', 'read_sql',
                   'read_pickle', 'system', 'remove', 'unlink', 'rmtree'}


def table_schema(tables):
    parts = []
    for name, df in tables.items():
        cols = ', '.join(f'{c} ({t})' for c, t in df.dtypes.astype(str).items())
        sample = df.head(3).to_csv(index=False)
        parts.append(f'DataFrame `{name}`: {len(df):,} rows\nColumns: {cols}\nSample:\n{sample}')
    return '\n'.join(parts)


def check_code_safety(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f'Синтаксична помилка: {e}'
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return 'Імпорти заборонені'
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            return f'Заборонене ім\'я: {node.id}'
        if isinstance(node, ast.Attribute):
            if node.attr.startswith('__') or node.attr in FORBIDDEN_ATTRS:
                return f'Заборонений атрибут: {node.attr}'
    return None


def run_code(code, tables):
    namespace = {'pd': pd, 'np': np, **{k: v.copy() for k, v in tables.items()}}
    exec(code, {'__builtins__': {'len': len, 'range': range, 'min': min, 'max': max, 'sum': sum,
                                 'round': round, 'abs': abs, 'sorted': sorted, 'list': list,
                                 'dict': dict, 'str': str, 'int': int, 'float': float,
                                 'zip': zip, 'enumerate': enumerate, 'set': set, 'tuple': tuple,
                                 'bool': bool, 'any': any, 'all': all,
                                 'isinstance': isinstance, 'print': print, 'map': map, 'reversed': reversed}}, namespace)
    if 'result' not in namespace:
        raise ValueError('Код не створив змінну result')
    return namespace['result']


def ask(question, tables=None, interpret=True):
    tables = TABLES if tables is None else tables
    display(Markdown(f'### ❓ {question}'))
    if not question or not question.strip():
        display(Markdown('❌ Порожнє питання.'))
        return None
    system = NL2PANDAS_SYSTEM.replace('{schema}', table_schema(tables))
    plan = call_llm(system, f'Question: {question}')

    if not plan.get('can_answer', False) or not plan.get('code'):
        display(Markdown(f'🚫 **Асистент не може відповісти на це питання за даними:** {plan.get("explanation", "")}'))
        return plan

    code = plan['code']
    for attempt in range(2):
        display(Markdown(f'**Згенерований код{" (виправлений)" if attempt else ""}:**\n```python\n{code}\n```'))
        problem = check_code_safety(code)
        if problem:
            display(Markdown(f'⛔ **Код не виконано з міркувань безпеки:** {problem}'))
            return {**plan, 'blocked': problem}
        try:
            result = run_code(code, tables)
            break
        except Exception as e:
            error = f'{type(e).__name__}: {e}'
            display(Markdown(f'⚠️ Помилка виконання: `{error}`'))
            if attempt == 1:
                return {**plan, 'error': error}
            fix = call_llm(system, f'Question: {question}\nYour code:\n{code}\nIt failed with: {error}\n'
                                   f'Return corrected JSON in the same schema.')
            code = fix.get('code') or code

    display(Markdown(f'**Що робить код:** {plan.get("explanation", "")}'))
    if plan.get('assumptions'):
        display(Markdown('**Припущення:** ' + '; '.join(plan['assumptions'])))
    display(result if isinstance(result, (pd.DataFrame, pd.Series)) else Markdown(f'**Результат:** `{result}`'))

    answer = None
    if interpret:
        preview = result.head(20).to_string() if isinstance(result, (pd.DataFrame, pd.Series)) else str(result)
        answer = call_llm(INTERPRET_SYSTEM, f'Question: {question}\nResult:\n{preview}').get('answer')
        display(Markdown(f'**Відповідь:** {answer}'))
        show_check(answer, preview)
    return {'code': code, 'result': result, 'answer': answer}

In [13]:
r1 = ask('Покажи топ-10 країн за виручкою і кількістю платників')

### ❓ Покажи топ-10 країн за виручкою і кількістю платників

**Згенерований код:**
```python
result = (
    users.groupby(['country_code', 'country_name'], dropna=False, as_index=False)
    .agg(total_revenue=('revenue', 'sum'), payers=('is_payer', 'sum'))
    .sort_values(['total_revenue', 'payers'], ascending=[False, False])
    .head(10)
)
```

**Що робить код:** Код групує користувачів за кодом та назвою країни, підсумовує їхню виручку (`revenue`) та кількість платників (`is_payer`), після чого сортує за спаданням виручки та кількості платників і повертає топ-10 країн.

**Припущення:** Використано таблицю `users`, де вже агреговано загальну виручку (`revenue`) та ознаку платника (`is_payer`) для кожного користувача.; Сортування для визначення топ-10 виконано насамперед за сумарною виручкою (`total_revenue`), а у разі рівності — за кількістю платників (`payers`).

,country_code,country_name,total_revenue,payers
180,UA,Ukraine,"22,954.52",2185
182,US,United States of America,"19,141.17",1213
193,ZA,South Africa,"6,053.40",620
35,CL,Chile,"5,513.70",455
53,ES,Spain,"4,895.21",387
60,GB,United Kingdom of Great Britain and Northern I...,"3,966.89",315
101,KZ,Kazakhstan,"3,669.68",352
10,AR,Argentina,"2,957.92",257
45,DE,Germany,"2,689.54",167
144,PL,Poland,"2,210.52",153


**Відповідь:** Лідерами рейтингу за виручкою та кількістю платників є Україна (22,954.52 та 2185 платників) і США (19,141.17 та 1213 платників). Далі за рівнем виручки йдуть Південна Африка (6,053.40; 620 платників), Чилі (5,513.70; 455), Іспанія (4,895.21; 387) та Велика Британія (3,966.89; 315). Замикають десятку Казахстан (3,669.68; 352), Аргентина (2,957.92; 257), Німеччина (2,689.54; 167) та Польща (2,210.52; 153).

✅ **Перевірка чисел:** усі числа в тексті знайдено серед пораховано даних.

In [14]:
r2 = ask('Яка конверсія в оплату (частка платників) на кожній платформі серед нових користувачів?')

### ❓ Яка конверсія в оплату (частка платників) на кожній платформі серед нових користувачів?

**Згенерований код:**
```python
result = (
    users[users['is_new']]
    .groupby('app', as_index=False, dropna=False)
    .agg(
        new_users=('user_id', 'count'),
        payers=('is_payer', 'sum'),
        conversion_rate=('is_payer', 'mean')
    )
    .sort_values('conversion_rate', ascending=False)
)
```

**Що робить код:** Код фільтрує таблицю users за умовою is_new == True, групує нових користувачів за платформою (app) та обчислює кількість користувачів, кількість платників і конверсію в оплату (середнє значення is_payer).

**Припущення:** Платформа користувача визначається колонкою app у таблиці users.; До нових користувачів належать записи, для яких is_new дорівнює True.; Конверсія в оплату обчислюється як частка платників (is_payer == True) серед усіх нових користувачів відповідної платформи.

,app,new_users,payers,conversion_rate
0,android,854,125,0.15
1,desktop,1935,188,0.10
3,mobile,128205,5906,0.05
2,ios,8799,390,0.04


**Відповідь:** Найвища конверсія в оплату серед нових користувачів зафіксована на платформі android і становить 0.15 (125 платників із 854 нових користувачів), а на платформі desktop цей показник дорівнює 0.10 (188 із 1935). На платформі mobile конверсія складає 0.05 (5906 платників зі 128205 нових користувачів). Найнижчу конверсію має платформа ios — 0.04 (390 платників із 8799 нових користувачів).

✅ **Перевірка чисел:** усі числа в тексті знайдено серед пораховано даних.

In [15]:
r3 = ask('Скільки в середньому сесій і хвилин на користувача у платників і неплатників?')

### ❓ Скільки в середньому сесій і хвилин на користувача у платників і неплатників?

**Згенерований код:**
```python
result = (
    users.assign(
        n_sessions=users['n_sessions'].fillna(0),
        total_duration_min=users['total_duration_min'].fillna(0)
    )
    .groupby('is_payer', as_index=False)[['n_sessions', 'total_duration_min']]
    .mean()
    .rename(columns={
        'n_sessions': 'avg_sessions',
        'total_duration_min': 'avg_duration_min'
    })
)
```

**Що робить код:** Код групує таблицю users за ознакою платника (is_payer) та обчислює середню кількість сесій і загальну тривалість у хвилинах на одного користувача.

**Припущення:** Для користувачів, які не мали жодної сесії (значення NaN у стовпцях n_sessions або total_duration_min), пропуски замінено на 0, щоб врахувати всіх зареєстрованих користувачів.; Статус платника визначається за булевим стовпцем is_payer у таблиці users.

,is_payer,avg_sessions,avg_duration_min
0,False,6.85,199.97
1,True,22.49,828.76


**Відповідь:** На одного користувача-платника в середньому припадає 22.49 сесій та 828.76 хвилин. У неплатників ці показники значно нижчі й становлять у середньому 6.85 сесій та 199.97 хвилин на користувача. Таким чином, платна аудиторія демонструє суттєво вищу залученість за обома метриками.

✅ **Перевірка чисел:** усі числа в тексті знайдено серед пораховано даних.

## Функція 4. Пояснення зміни метрики
На вхід: метрика і два періоди. **Pandas рахує** значення метрики в кожному сегменті (платформа, країна, стать, вік) в обох періодах і **внесок кожного сегмента** у загальну зміну.
Якщо періоди різної довжини, функція попереджає і **порівнює середнє за день**, бо інакше неповний тиждень виглядатиме як падіння.
**LLM формулює гіпотезу причини** лише на основі найбільших внесків; згадані сегменти звіряються з даними.

In [16]:
CHANGE_SYSTEM = """You are a product analyst explaining why a metric changed between two periods.
You receive a pandas decomposition: total change and the segments with the largest contribution.
Rules:
- Use only numbers from the input. Do not invent segments that are not in the input.
- Distinguish real business change from artifacts (unequal period length, tracking changes, data cut-off).
- Write in Ukrainian.
Return JSON: {"headline": "one sentence", "drivers": [{"dimension": "...", "segment": "...", "explanation": "..."}],
 "hypotheses": ["possible causes, 2-4"], "next_checks": ["what to verify, 2-3"]}"""

METRICS = {
    'revenue':      ('payments', 'pay_dt', lambda d: d['amount_usd'].sum()),
    'payers':       ('payments', 'pay_dt', lambda d: d['user_id'].nunique()),
    'new_users':    ('registrations', 'reg_dt', lambda d: d['user_id'].nunique()),
    'active_users': ('sessions', 'session_start_dt', lambda d: d['user_id'].nunique()),
}


def _period(df, col, start, end):
    return df[(df[col] >= pd.Timestamp(start)) & (df[col] < pd.Timestamp(end) + pd.Timedelta(days=1))]


def explain_change(metric, previous, current, dims=('app', 'country_name', 'gender', 'age_group'), top=8):
    if metric not in METRICS:
        display(Markdown(f'❌ Невідома метрика `{metric}`. Доступні: {list(METRICS)}'))
        return None
    table, date_col, fn = METRICS[metric]
    base = TABLES[table]
    need = [c for c in ['app', 'country_name', 'gender', 'age_group'] if c not in base.columns]
    base = base.merge(users[['user_id'] + need], on='user_id', how='left') if need else base
    prev, cur = _period(base, date_col, *previous), _period(base, date_col, *current)
    if len(prev) == 0 or len(cur) == 0:
        display(Markdown('❌ В одному з періодів немає даних. Перевір дати.'))
        return None

    days_prev = (pd.Timestamp(previous[1]) - pd.Timestamp(previous[0])).days + 1
    days_cur = (pd.Timestamp(current[1]) - pd.Timestamp(current[0])).days + 1
    v_prev, v_cur = fn(prev), fn(cur)
    summary = {'metric': metric, 'previous_period': list(previous), 'current_period': list(current),
               'days_previous': days_prev, 'days_current': days_cur,
               'value_previous': round(float(v_prev), 2), 'value_current': round(float(v_cur), 2),
               'change_pct': round(float((v_cur / v_prev - 1) * 100), 2) if v_prev else None,
               'per_day_previous': round(float(v_prev / days_prev), 2),
               'per_day_current': round(float(v_cur / days_cur), 2)}
    summary['per_day_change_pct'] = round((summary['per_day_current'] / summary['per_day_previous'] - 1) * 100, 2)

    display(Markdown(f'### Зміна `{metric}`: {previous[0]}…{previous[1]} → {current[0]}…{current[1]}'))
    if days_prev != days_cur:
        display(Markdown(f'⚠️ Періоди різної довжини ({days_prev} і {days_cur} дн.). Тому порівнюємо **середнє за день**: '
                         f'{summary["per_day_previous"]:,.2f} → {summary["per_day_current"]:,.2f} '
                         f'({summary["per_day_change_pct"]:+.1f}%), а не суму ({summary["change_pct"]:+.1f}%).'))

    rows = []
    for dim in dims:
        for seg in set(prev[dim].dropna().unique()) | set(cur[dim].dropna().unique()):
            a = fn(prev[prev[dim] == seg]) / days_prev
            b = fn(cur[cur[dim] == seg]) / days_cur
            rows.append({'dimension': dim, 'segment': str(seg), 'per_day_previous': round(float(a), 2),
                         'per_day_current': round(float(b), 2), 'delta_per_day': round(float(b - a), 2)})
    seg = pd.DataFrame(rows)
    total_delta = summary['per_day_current'] - summary['per_day_previous']
    seg['share_of_change_pct'] = (seg['delta_per_day'] / total_delta * 100).round(1) if total_delta else 0.0
    top_seg = seg.reindex(seg['delta_per_day'].abs().sort_values(ascending=False).index).head(top)
    display(pd.DataFrame([summary]).T.rename(columns={0: 'значення'}))
    display(top_seg)

    report = call_llm(CHANGE_SYSTEM, json.dumps({'summary': summary, 'top_segments': top_seg.to_dict('records')},
                                                ensure_ascii=False, default=str))
    valid = set(zip(seg['dimension'], seg['segment']))
    md = [f'**{report.get("headline", "")}**', '', '**Основні драйвери:**']
    for d in report.get('drivers', []):
        mark = '' if (d.get('dimension'), str(d.get('segment'))) in valid else ' ⚠️ *(сегмента немає в даних)*'
        md.append(f'- `{d.get("dimension")}={d.get("segment")}`{mark}: {d.get("explanation", "")}')
    md += ['', '**Можливі причини:**'] + [f'- {h}' for h in report.get('hypotheses', [])]
    md += ['', '**Що перевірити:**'] + [f'- {c}' for c in report.get('next_checks', [])]
    display(Markdown('\n'.join(md)))
    show_check(json.dumps(report, ensure_ascii=False), {'s': summary, 'seg': seg.to_dict('records')})
    return {'summary': summary, 'segments': seg, 'report': report}

In [17]:
# Чому «впала» виручка останнього тижня? (тиждень неповний: дані закінчуються 11.05)
chg = explain_change('revenue', previous=('2024-04-29', '2024-05-05'), current=('2024-05-06', '2024-05-11'))

### Зміна `revenue`: 2024-04-29…2024-05-05 → 2024-05-06…2024-05-11

⚠️ Періоди різної довжини (7 і 6 дн.). Тому порівнюємо **середнє за день**: 4,063.34 → 3,250.03 (-20.0%), а не суму (-31.4%).

,значення
metric,revenue
previous_period,"[2024-04-29, 2024-05-05]"
current_period,"[2024-05-06, 2024-05-11]"
days_previous,7
days_current,6
value_previous,"28,443.38"
value_current,"19,500.19"
change_pct,-31.44
per_day_previous,"4,063.34"
per_day_current,"3,250.03"


,dimension,segment,per_day_previous,per_day_current,delta_per_day,share_of_change_pct
100,gender,m,"4,063.34","3,250.03",-813.31,100.00
2,app,mobile,"3,805.01","3,125.68",-679.33,83.50
101,age_group,25-34,"1,103.26",848.88,-254.39,31.30
103,age_group,45-54,785.74,554.60,-231.14,28.40
102,age_group,18-24,628.88,465.23,-163.65,20.10
43,country_name,South Africa,305.30,181.02,-124.28,15.30
54,country_name,Spain,276.41,155.55,-120.86,14.90
104,age_group,35-44,"1,164.98","1,061.64",-103.34,12.70


**Загальний дохід (revenue) знизився на -31.44% (з 28443.38 до 19500.19) частково через нерівну тривалість періодів (6 днів проти 7), але реальне середньоденне падіння також суттєве і становить -20.02% (з 4063.34 до 3250.03 на день).**

**Основні драйвери:**
- `gender=m`: Повністю формує весь зафіксований дохід і 100.0% його зміни: середньоденне значення впало з 4063.34 до 3250.03 (дельта -813.31 на день).
- `app=mobile`: Основний драйвер падіння за платформою, що забезпечує 83.5% зміни: дохід на день знизився з 3805.01 до 3125.68 (дельта -679.33).
- `age_group=25-34`: Найбільший внесок серед вікових груп (31.3% зміни): середньоденний дохід скоротився з 1103.26 до 848.88 (дельта -254.39).
- `age_group=45-54`: Другий за масштабом драйвер серед вікових груп (28.4% зміни): падіння з 785.74 до 554.6 на день (дельта -231.14).
- `age_group=18-24`: Забезпечує 20.1% зміни: дохід на день зменшився з 628.88 до 465.23 (дельта -163.65).
- `country_name=South Africa`: Найбільше зниження серед країн (15.3% зміни): середньоденний дохід впав з 305.3 до 181.02 (дельта -124.28).
- `country_name=Spain`: Забезпечує 14.9% загальної зміни: падіння з 276.41 до 155.55 на день (дельта -120.86).

**Можливі причини:**
- Артефакт порівняння періодів різної довжини: поточний період 2024-05-06 — 2024-05-11 охоплює 6 днів проти 7 днів у попередньому (2024-04-29 — 2024-05-05), що штучно поглиблює сумарне падіння до -31.44%, хоча реальне зниження за денним показником також становить -20.02%.
- Технічний збій, невдалий реліз або погіршення платіжної конверсії саме в додатку mobile, на який припадає 83.5% денного зниження (-679.33).
- Проблеми з локальними платіжними методами, ціноутворенням або завершення маркетингової активності у країнах South Africa (-124.28 на день) та Spain (-120.86 на день), а також зниження активності в ключових вікових групах 25-34, 45-54 та 18-24.
- Особливість продукту або помилка трекінгу атрибута gender, оскільки 100.0% доходу в обидва періоди (4063.34 та 3250.03 на день) припадає виключно на сегмент m.

**Що перевірити:**
- Перевірити, чи відсутність останнього дня в періоді 2024-05-06 — 2024-05-11 є затримкою надходження даних (cut-off), та порівняти однакові за тривалістю та днями тижня періоди.
- Проаналізувати технічні метрики, краші та воронку оплат у додатку mobile, окремо перевіривши транзакції в South Africa та Spain і серед вікових груп 25-34, 45-54 та 18-24.
- Перевірити коректність логування поля gender, щоб упевнитися, чи немає помилки атрибуції для сегмента m.

✅ **Перевірка чисел:** усі числа в тексті знайдено серед пораховано даних.

## Функція 5. Executive summary
**Pandas рахує набір KPI** за період і зміну до попереднього періоду. **LLM пише короткий бізнес-текст:** що сталося, чому це важливо, що робити далі.
Після цього **кожне число в тексті звіряється з KPI**. Якщо модель написала число, якого немає в даних, воно позначається ⚠️.

In [18]:
SUMMARY_SYSTEM = """You write an executive summary for a product manager.
You receive KPIs computed with pandas for the current and previous period.
Rules:
- Use ONLY numbers from the KPI JSON (you may round them). Never compute new numbers.
- Business language, no jargon. Max 120 words in total.
- Structure: what happened -> why it matters -> what to do next.
- Write in Ukrainian.
Return JSON: {"headline": "one sentence", "what_happened": ["2-4 bullets"], "why_it_matters": "1-2 sentences", "next_steps": ["2-3 bullets"]}"""


def compute_kpis(start, end, prev_start, prev_end):
    def block(a, b):
        reg = _period(registrations, 'reg_dt', a, b)
        pay = _period(payments, 'pay_dt', a, b)
        ses = _period(sessions, 'session_start_dt', a, b)
        days = (pd.Timestamp(b) - pd.Timestamp(a)).days + 1
        k = {'days': days, 'new_users': int(reg['user_id'].nunique()),
             'active_users': int(ses['user_id'].nunique()),
             'avg_dau': round(float(ses.groupby(ses['session_start_dt'].dt.date)['user_id'].nunique().mean()), 1) if len(ses) else 0,
             'revenue_usd': round(float(pay['amount_usd'].sum()), 2),
             'payers': int(pay['user_id'].nunique()),
             'avg_check_usd': round(float(pay['amount_usd'].mean()), 2) if len(pay) else 0}
        k['revenue_per_day_usd'] = round(k['revenue_usd'] / days, 2)
        return k
    cur, prev = block(start, end), block(prev_start, prev_end)
    change = {k: round((cur[k] / prev[k] - 1) * 100, 1) for k in cur if k != 'days' and prev[k]}
    top_country = (_period(payments, 'pay_dt', start, end).merge(users[['user_id', 'country_name']], on='user_id')
                   .groupby('country_name')['amount_usd'].sum().nlargest(3).round(2).to_dict())
    return {'current_period': [start, end], 'previous_period': [prev_start, prev_end],
            'current': cur, 'previous': prev, 'change_pct': change, 'top3_countries_revenue_usd': top_country}


def executive_summary(kpis):
    display(Markdown('**KPI (пораховано Pandas):**'))
    display(pd.DataFrame({'поточний': kpis['current'], 'попередній': kpis['previous'],
                          'зміна, %': pd.Series(kpis['change_pct'])}))
    s = call_llm(SUMMARY_SYSTEM, json.dumps(kpis, ensure_ascii=False))
    text = '\n'.join([f'## {s.get("headline", "")}', '', '**Що сталося**'] +
                     [f'- {x}' for x in s.get('what_happened', [])] +
                     ['', f'**Чому це важливо.** {s.get("why_it_matters", "")}', '', '**Що робити далі**'] +
                     [f'- {x}' for x in s.get('next_steps', [])])
    display(Markdown(text))
    show_check(json.dumps(s, ensure_ascii=False), kpis)
    return s

In [19]:
kpis = compute_kpis('2024-04-22', '2024-04-28', '2024-04-15', '2024-04-21')
summary = executive_summary(kpis)

**KPI (пораховано Pandas):**

,поточний,попередній,"зміна, %"
active_users,"85,088.00","54,444.00",56.30
avg_check_usd,8.10,8.66,-6.50
avg_dau,"31,840.60","13,369.40",138.20
days,7.00,7.00,NaN
new_users,"31,879.00","20,108.00",58.50
payers,"2,259.00","1,129.00",100.10
revenue_per_day_usd,"3,574.58","1,996.10",79.10
revenue_usd,"25,022.06","13,972.71",79.10


## За тиждень дохід зріс на 79,1% завдяки притоку користувачів і подвоєнню кількості платників.

**Що сталося**
- Нові користувачі зросли на 58,5% до 31 879, активна аудиторія — на 56,3% до 85 088.
- Щоденна аудиторія піднялася на 138,2% (до 31 841), а платники — на 100,1% (до 2 259).
- Дохід сягнув 25 022 доларів попри падіння середнього чеку на 6,5% (до 8,1 долара); лідирує Україна (6 191 долар).

**Чому це важливо.** Продукт ефективно залучає та щодня утримує платоспроможну аудиторію. Проте падіння середнього чеку вказує на зміщення попиту до дешевших покупок.

**Що робити далі**
- Визначити причини зниження середнього чеку та протестувати пропозиції для його підвищення.
- Масштабувати канали залучення з України та США (4 530 доларів доходу).

⚠️ **Перевірка чисел:** числа, яких немає в даних: `[6.5]`. Їх треба перевірити вручну.

## 6. Крайні випадки
Перевіряємо, як асистент поводиться з некоректними вхідними даними.

In [20]:
auto_eda(registrations.iloc[0:0], 'порожня таблиця')            # порожні дані → без виклику LLM

### EDA: `порожня таблиця`
❌ Таблиця порожня, аналізувати нічого. LLM не викликається.

In [21]:
ask('Яка зараз погода в Києві?')                                  # питання не про дані

### ❓ Яка зараз погода в Києві?

🚫 **Асистент не може відповісти на це питання за даними:** Неможливо відповісти на це запитання, оскільки воно стосується поточної погоди в Києві, а доступні таблиці містять лише аналітичні дані про користувачів, реєстрації, сесії, платежі та країни без жодної інформації про погоду чи доступу до мережі.

{'can_answer': False,
 'code': '',
 'explanation': 'Неможливо відповісти на це запитання, оскільки воно стосується поточної погоди в Києві, а доступні таблиці містять лише аналітичні дані про користувачів, реєстрації, сесії, платежі та країни без жодної інформації про погоду чи доступу до мережі.',
 'assumptions': []}

In [22]:
ask('Видали всі файли в поточній папці і покажи список файлів')     # спроба небезпечної дії

### ❓ Видали всі файли в поточній папці і покажи список файлів

🚫 **Асистент не може відповісти на це питання за даними:** Запит не стосується аналізу даних у наданих таблицях, а вимагає видалення файлів в ОС, що заборонено правилами безпеки.

{'can_answer': False,
 'code': '',
 'explanation': 'Запит не стосується аналізу даних у наданих таблицях, а вимагає видалення файлів в ОС, що заборонено правилами безпеки.',
 'assumptions': ['Користувач запитує системну операцію видалення файлів, яка не може бути виконана через pandas.']}

In [23]:
explain_change('revenue', previous=('2023-01-01', '2023-01-07'), current=('2024-04-22', '2024-04-28'))  # немає даних у періоді

❌ В одному з періодів немає даних. Перевір дати.

## 7. Журнал викликів LLM
Для кожного виклику видно модель, час і розмір промпта. Повні промпти й відповіді лежать у `LLM_LOG`.

In [24]:
pd.DataFrame([{'#': i + 1, 'model': c['model'], 'seconds': c['seconds'],
               'prompt_chars': len(c['system']) + len(c['user']),
               'response_keys': ', '.join(c['response'].keys()) if isinstance(c['response'], dict) else ''}
              for i, c in enumerate(LLM_LOG)])

,#,model,seconds,prompt_chars,response_keys
0,1,gemini-4-hydrogen,55.40,40,status
1,2,gemini-4-hydrogen,130.00,2230,"summary, issues, observations, next_steps"
2,3,gemini-4-hydrogen,20.30,1871,"summary, issues, observations, next_steps"
3,4,gemini-4-hydrogen,60.40,15035,hypotheses
4,5,gemini-4-hydrogen,18.90,3036,"can_answer, code, explanation, assumptions"
5,6,gemini-4-hydrogen,31.70,1309,answer
6,7,gemini-4-hydrogen,51.70,3070,"can_answer, code, explanation, assumptions"
7,8,gemini-4-hydrogen,73.80,544,answer
8,9,gemini-4-hydrogen,27.20,3060,"can_answer, code, explanation, assumptions"
9,10,gemini-4-hydrogen,20.30,431,answer
